# GridPilot-RL: Reinforcement Learning for Smart Grid Energy Management

GPU/CPU benchmark notebook for **Google Colab** or **Kaggle**.

**Novel Contributions:**
- **Physics-Guided Residual RL (`Residual-PPO`, `Residual-SAC`)** with `VecNormalize`, engineered net-load & price-spread features, and `gamma=0.995`
- **Analytical Physics Safety Shield** (SoC limits, inverter ramp-rate bounds, pre-peak SoC reserve)
- **Uncertainty-Aware Multi-Horizon Forecasting** (`H=6` look-ahead `mu + sigma`)
- **Non-Linear Electrochemical Aging, Carbon Co-Optimization, CVaR-20% Tail Risk & Exact HiGHS LP Oracle** on **real OPSD + Fraunhofer Energy-Charts data**

In [ ]:
!pip install -q stable-baselines3 gymnasium pandas numpy matplotlib scipy torch

In [ ]:
import torch, os
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Option A: If running from a cloned GitHub repository:
# !git clone https://github.com/<your-username>/GridPilot-RL.git
# %cd GridPilot-RL

# Option B: If uploaded as GridPilot-RL.zip to Colab/Kaggle working directory, unzip and enter
!if [ -f GridPilot-RL.zip ]; then unzip -qo GridPilot-RL.zip; fi
if os.path.exists('GridPilot-RL'):
    %cd GridPilot-RL
!pip install -e . --no-deps
!python -m pytest -v

In [ ]:
# High-budget GPU/Colab benchmark (150k main steps, 60k ablation steps, 20 held-out test episodes)
!python -u scripts/run_benchmark.py --timesteps 150000 --ablation-timesteps 60000 --n-eval-episodes 20

In [ ]:
import pandas as pd
from IPython.display import Image, display

summary = pd.read_csv('results/tables/eval_summary_main.csv')
print('=== Held-Out Real OPSD Results (nominal_real) ===')
display(summary[summary['scenario'] == 'nominal_real'][[
    'policy', 'total_energy_cost_mean', 'peak_demand_kw_mean',
    'battery_degradation_cost_mean', 'total_carbon_kg_mean',
    'total_operating_cost_mean', 'cvar20_operating_cost',
    'operating_cost_reduction_pct', 'shield_intervention_rate_mean'
]].sort_values('total_operating_cost_mean').round(3))

ablation = pd.read_csv('results/tables/forecast_ablation.csv')
print('=== Forecast & Uncertainty Ablation ===')
display(ablation.groupby(['algo', 'forecast_mode'], as_index=False)[[
    'total_energy_cost_mean', 'peak_demand_kw_mean', 'total_operating_cost_mean', 'cvar20_operating_cost'
]].mean().round(3))

for fig in [
    'training_curves.png',
    'multi_objective_nominal.png',
    'robustness_scenarios.png',
    'pareto_cost_degradation.png',
    'dispatch_48h_real.png',
    'forecast_ablation.png',
]:
    display(Image(filename=f'results/figures/{fig}'))